<a href="https://colab.research.google.com/github/kalyan-1845/flyrank-ml-assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kalyan-1845/flyrank-ml-assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import duckdb
from google.colab import userdata
import pandas as pd

# Connect to Hugging Face
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# We are building features from Jan 31 to March 31, to predict what happens in April
query = f"""
    WITH windowed AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date > '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN report_date <= '2026-02-28' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > '2026-02-28' THEN gsc_avg_position END) AS pos_last30
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-0*/*.parquet')
        WHERE report_date >= '2026-01-31' AND report_date <= '2026-03-31'
        GROUP BY content_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
    LIMIT 5
"""
features = con.sql(query).df()

# Fill any missing position data with '100' (meaning it didn't rank on Google at all)
features['pos_last30'] = features['pos_last30'].fillna(100)
print(features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
imp_last30: Total impressions in the 30 days right before the decision. Missing values are 0. Available BEFORE prediction? Yes.
imp_prev30: Total impressions in the 30-day window before that (days 31-60). Missing values are 0. Available BEFORE prediction? Yes.
pos_last30: Average ranking position in the last 30 days. Missing values are filled with 100 (unranked). Available BEFORE prediction? Yes.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- THE LEAKAGE TEST ---")
print("We are predicting what happens in April. If we accidentally include April's impressions as a feature, the model cheats and gets 100% accuracy.")

leakage_query = f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date >= '2026-04-01' THEN gsc_impressions ELSE 0 END) AS imp_future_LEAK
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
    LIMIT 5
"""
leakage_df = con.sql(leakage_query).df()
print("\nWARNING: This column 'imp_future_LEAK' uses data from April. It must be dropped before training!")
print(leakage_df)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
I explicitly excluded client_hash_id and the specific Page URLs. Why: These are unique identifiers. If we include them, the AI will just memorize the specific clients in the training data instead of learning universal patterns. It will completely fail when tested on a brand new client.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.